# FloodRisk - Linear Regression & MAE Evaluation

FloodRisk is primarily a classification problem.

This notebook demonstrates Linear Regression concepts separately using a continuous target variable.

In [43]:
#Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [44]:
#Load Dataset

df = pd.read_csv('../data/raw/flood_data.csv')

df.head()

,rainfall,temperature,humidity,river_level,soil_moisture,flood_risk
0,23.5,18.2,45.0,1.2,0.35,0
1,34.2,22.1,52.0,1.8,0.42,0
2,45.6,25.3,65.0,2.5,0.48,1
3,28.3,20.5,48.0,1.5,0.38,0
4,52.1,28.1,72.0,3.2,0.55,1


In [45]:
#Select Regression Features and Target

X = df[['rainfall', 'humidity', 'temperature']]
y = df['river_level']

In [46]:
#Train-test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [47]:
# Baseline Model Using DummyRegressor

baseline = DummyRegressor(strategy='mean')

baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)

In [48]:
#Train Linear Regression Model

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)

predictions = pipeline.predict(X_test)

In [49]:
#Evaluate Model

print("===== LINEAR REGRESSION =====")

print("RMSE:",
      np.sqrt(mean_squared_error(y_test, predictions)))

print("MAE:",
      mean_absolute_error(y_test, predictions))

print("R2:",
      r2_score(y_test, predictions))

===== LINEAR REGRESSION =====
RMSE: 0.07153821557110314
MAE: 0.05326717621030372
R2: 0.9912246170855821


In [50]:
#Baseline Comparison

baseline_mae = mean_absolute_error(y_test, baseline_preds)

model_mae = mean_absolute_error(y_test, predictions)

print("Baseline MAE:", baseline_mae)

print("Linear Regression MAE:", model_mae)

improvement = baseline_mae - model_mae

print("MAE Improvement:", improvement)

Baseline MAE: 0.6894999999999999
Linear Regression MAE: 0.05326717621030372
MAE Improvement: 0.6362328237896961


In [51]:
#Coefficient Interpretation

model = pipeline.named_steps['model']

coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

print(coefficients)

       Feature  Coefficient
0     rainfall     0.264390
1     humidity     0.797811
2  temperature    -0.186865


In [52]:
# Cross Validation Using MAE

from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='neg_mean_absolute_error'
)

cv_mae_scores = -cv_scores

print("CV MAE Scores:")
print(cv_mae_scores)

print()

print("Mean CV MAE:",
      cv_mae_scores.mean())

print("Std CV MAE:",
      cv_mae_scores.std())

CV MAE Scores:
[0.04232309 0.04811692 0.03798717 0.04773164 0.06535324]

Mean CV MAE: 0.048302412037471106
Std CV MAE: 0.009308569172996856


# Conclusion

Linear Regression was successfully trained and evaluated using MAE, RMSE, and R² metrics.

The model outperformed the baseline DummyRegressor, indicating that environmental variables contain predictive information for river level estimation.

Cross-validation showed relatively stable MAE scores across folds.